In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import scanpy as sc
import episcanpy as epi
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

import glob
import os

from pathlib import Path
from scipy import io
from helps import *

# Raw CM

## Peak

In [ ]:
fr_file="Datasets/HSPC/RawFiles/GSM6403411_3423-MV-2_atac_fragments.tsv.gz"
metadata=sc.read_h5ad("Datasets/HSPC/RawFiles/3423-MV-2_adata_postpro.h5ad").obs
metadata=pd.DataFrame(metadata["leiden"])
metadata.rename({"leiden" : "CellType"}, axis=1, inplace=True)
features_space="Peak"
features_file="Datasets/HSPC/RawFiles/HSPC_PeakCalling/HSPC_peaks.narrowPeak"
adata=create_count_matrix(fragments_file=fr_file, valid_bcs=list(set(metadata.index)), features_space=features_space,
                          features_file=features_file, metadata=metadata)
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())

In [ ]:
epi.pp.nucleosome_signal(adata,fr_file, n=10000)
epi.pp.tss_enrichment(adata, gtf="../AnnotRef/hs/hg38/gencode.v44.annotation.gtf.gz", source="HAVANA",
                      fragments=fr_file, n=5000, score="avg_score_of_center_region", distance_to_tss=1000, bp_per_flank=100)
Path("Datasets/HSPC/CM").mkdir(parents=True, exist_ok=True)
adata.write("Datasets/HSPC/CM/HSPC_Peak_Raw.h5ad", compression="gzip")

## GEX

In [ ]:
adata=sc.read_10x_mtx("Datasets/HSPC/RawFiles/gexmatrix", prefix="GSE209878_3423-MV-2_")
metadata=sc.read_h5ad("Datasets/HSPC/RawFiles/3423-MV-2_adata_postpro.h5ad").obs
metadata=pd.DataFrame(metadata["leiden"])
metadata.rename({"leiden" : "CellType"}, axis=1, inplace=True)

In [ ]:
adata=adata[intersection([adata.obs.index, metadata.index])]
adata.obs=metadata.loc[adata.obs.index]
sc.pp.scrublet(adata)

In [ ]:
adata.var_names_make_unique()

In [ ]:
adata.write("Datasets/HSPC/CM/HSPC_GEX_Raw.h5ad", compression="gzip")

## mRNA

In [ ]:
adata=sc.read_h5ad("Datasets/HSPC/CM/HSPC_GEX_Raw.h5ad")
names = ["chr", "source", "type", "start", "stop", "score", "strand", "frame", "attribute"]
gtf_file="../AnnotRef/hs/hg38/gencode.v44.annotation.gtf.gz"
gtf=pd.read_csv(gtf_file, sep="\t", header=None, comment="#", names=names, dtype={"chr": str})
gtf=gtf[(gtf["type"]=="gene") & (gtf["source"]=="HAVANA")]

gene_names=[attr.replace("gene_name", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_name")]
tmp = [attr.replace("gene_type", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_type ")]
if not tmp:
    tmp = [attr.replace("gene_biotype", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_biotype ")]

info=pd.DataFrame(data=[gene_names,tmp], index=["GeneName","GeneType"]).T

mRNA=adata[:, intersection([adata.var.index, info[info["GeneType"]=="protein_coding"]["GeneName"]])]
mRNA.X=scipy.sparse.csr_matrix(mRNA.X.copy())
mRNA.layers["raw"]=scipy.sparse.csr_matrix(mRNA.X.copy())
mRNA.write("Datasets/HSPC/CM/HSPC_mRNA_Raw.h5ad", compression="gzip")

In [ ]:
mRNA

## lncRNA

In [ ]:
adata=sc.read_h5ad("Datasets/HSPC/CM/HSPC_GEX_Raw.h5ad")
names = ["chr", "source", "type", "start", "stop", "score", "strand", "frame", "attribute"]
gtf_file="../AnnotRef/hs/hg38/gencode.v44.annotation.gtf.gz"
gtf=pd.read_csv(gtf_file, sep="\t", header=None, comment="#", names=names, dtype={"chr": str})
gtf=gtf[(gtf["type"]=="gene") & (gtf["source"]=="HAVANA")]

gene_names=[attr.replace("gene_name", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_name")]
tmp = [attr.replace("gene_type", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_type ")]
if not tmp:
    tmp = [attr.replace("gene_biotype", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_biotype ")]

info=pd.DataFrame(data=[gene_names,tmp], index=["GeneName","GeneType"]).T

lncRNA=adata[:, intersection([adata.var.index, info[info["GeneType"]=="lncRNA"]["GeneName"]])]
lncRNA.X=scipy.sparse.csr_matrix(lncRNA.X.copy())
lncRNA.layers["raw"]=scipy.sparse.csr_matrix(lncRNA.X.copy())
lncRNA.write("Datasets/HSPC/CM/HSPC_lncRNA_Raw.h5ad", compression="gzip")

In [ ]:
lncRNA

# Processing

## Peak

In [ ]:
adata=sc.read_h5ad("Datasets/HSPC/CM/HSPC_Peak_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=False)

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.05)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
max_nucleosome_signal=2
epi.pl.fragment_length(adata, n=5000, threshold=max_nucleosome_signal, show_n=True)

In [ ]:
min_tss_enrichment = 4
adata.obs["tss"] = ["high" if val > min_tss_enrichment else "low" for val in adata.obs.tss_enrichment_score]

epi.pl.tss_enrichment(adata, group_by="tss", show_n=True)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata=adata[adata.obs.nucleosome_signal < max_nucleosome_signal]
adata=adata[adata.obs.tss_enrichment_score > min_tss_enrichment]
adata

In [ ]:
epi.pp.normalize_total(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')
min_var = np.quantile(adata.var.variances_norm, 0.9)
adata=adata[:, adata.var.variances_norm>min_var]
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/HSPC/CM/HSPC_Peak_Def.h5ad", compression="gzip")

## GEX

In [ ]:
adata=sc.read_h5ad("Datasets/HSPC/CM/HSPC_GEX_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=True)
mt_gene_mask = np.flatnonzero([gene.startswith(("MT","mt")) for gene in adata.var.index])
adata.obs['mt_frac'] = np.sum(adata[:, mt_gene_mask].X, axis=1).A1/adata.obs['n_counts']

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.05)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pl.histogram(adata, "mt_frac", max_threshold=0.05, show_log=False)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata = adata[adata.obs["mt_frac"]<=0.15]
adata=adata[adata.obs.predicted_doublet==False]
adata

In [ ]:
epi.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')
min_var = np.quantile(adata.var.variances_norm, 0.8)
adata=adata[:, adata.var.variances_norm>min_var]
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]

In [ ]:
adata

In [ ]:
adata.write("Datasets/HSPC/CM/HSPC_GEX_Def.h5ad", compression="gzip")

## mRNA

In [ ]:
adata=sc.read_h5ad("Datasets/HSPC/CM/HSPC_mRNA_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=True)
mt_gene_mask = np.flatnonzero([gene.startswith(("MT","mt")) for gene in adata.var.index])
adata.obs['mt_frac'] = np.sum(adata[:, mt_gene_mask].X, axis=1).A1/adata.obs['n_counts']

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.05)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
max_mt = np.quantile(adata.obs["mt_frac"].dropna(), 0.9)
epi.pl.histogram(adata, "mt_frac", max_threshold=max_mt, show_log=False)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata = adata[adata.obs["mt_frac"]<=0.05]
adata=adata[adata.obs.predicted_doublet==False]
adata

In [ ]:
epi.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')

In [ ]:
min_var = np.quantile(adata.var.variances_norm, 0.65)
adata=adata[:, adata.var.variances_norm>min_var]
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/HSPC/CM/HSPC_mRNA_Def.h5ad", compression="gzip")

## lncRNA

In [ ]:
adata=sc.read_h5ad("Datasets/HSPC/CM/HSPC_lncRNA_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=True)

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.05)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata=adata[adata.obs.predicted_doublet==False]
adata

In [ ]:
epi.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')

In [ ]:
min_var = np.quantile(adata.var.variances_norm, 0.15)
adata=adata[:, adata.var.variances_norm>min_var]
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/HSPC/CM/HSPC_lncRNA_Def.h5ad", compression="gzip")

## Muon objects

In [ ]:
import muon as mu
peak=sc.read_h5ad(f"Datasets/HSPC/CM/HSPC_Peak_Def.h5ad")
gex=sc.read_h5ad(f"Datasets/HSPC/CM/HSPC_GEX_Def.h5ad")
mrna=sc.read_h5ad(f"Datasets/HSPC/CM/HSPC_mRNA_Def.h5ad")
lncrna=sc.read_h5ad(f"Datasets/HSPC/CM/HSPC_lncRNA_Def.h5ad")

#Peak + GEX
print("Peak + GEX")
inter=intersection([peak.obs.index, gex.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "GEX" : gex[inter]})
mudata.write_h5mu(f"Datasets/HSPC/CM/HSPC_Peak_GEX_Def.h5mu", compression="gzip")

#Peak + mRNA
print("Peak + mRNA")
inter=intersection([peak.obs.index, mrna.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "mRNA" : mrna[inter]})
mudata.write_h5mu(f"Datasets/HSPC/CM/HSPC_Peak_mRNA_Def.h5mu", compression="gzip")

#Peak + lncRNA
print("Peak + lncRNA")
inter=intersection([peak.obs.index, lncrna.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/HSPC/CM/HSPC_Peak_lncRNA_Def.h5mu", compression="gzip")

#mRNA + lncRNA
print("mRNA + lncRNA")
inter=intersection([mrna.obs.index, lncrna.obs.index])
mudata=mu.MuData({"mRNA" : mrna[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/HSPC/CM/HSPC_mRNA_lncRNA_Def.h5mu", compression="gzip")

#Peak + mRNA + lncRNA
print("Peak + mRNA + lncRNA")
inter=intersection([peak.obs.index, mrna.obs.index, lncrna.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "mRNA" : mrna[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/HSPC/CM/HSPC_Peak_mRNA_lncRNA_Def.h5mu", compression="gzip")

# CTA

In [ ]:
df=sc.read_h5ad("Datasets/HSPC/CM/HSPC_GEX_Raw.h5ad").obs
df.head()

In [ ]:
df[["CellType"]].to_csv("Datasets/HSPC/HSPC_Metadata.tsv.gz", sep="\t", compression="gzip")